# Label Signal Boost Recipes

Use this notebook to rebuild stronger labels from the same candlestick images, then train on the new metadata.


In [ ]:
!pip install -q yfinance transformers

In [ ]:
from dataset_signal_boost import *
from kaggle_resnet_vit_boost import *

## Build Stronger Binary Labels

Start here. This drops ambiguous samples and usually gives a much cleaner learning target than the original ternary setup.


In [ ]:
signal_cfg = SignalConfig(
    data_root='/kaggle/input/stock-dataset/stock_dataset',
    output_csv='/kaggle/working/metadata_binary_1pct.csv',
    label_mode='binary_fixed',
    threshold=0.01,
    lookahead_days=1,
    purge_gap=19,
)
csv_path, summary_path = save_signal_metadata(signal_cfg)
print(csv_path)
print(summary_path)
pd.read_csv(csv_path)['label'].value_counts()

## Train ResNet50 On New Metadata

In [ ]:
train_cfg = TrainConfig(
    data_root='/kaggle/input/stock-dataset/stock_dataset',
    metadata_csv=str(csv_path),
    model_family='resnet50',
    class_names=('down', 'up'),
    image_size=320,
    batch_size=32,
    epochs=14,
    patience=4,
    purge_gap=19,
)
report = train_model(train_cfg)
plot_history(report)
plot_confusion(report)
pd.DataFrame(report['test']['classification_report']).T

## Train DeiT On The Same Metadata

Use this after the ResNet run. If GPU memory is tight, lower `batch_size` to `8`.


In [ ]:
vit_cfg = TrainConfig(
    data_root='/kaggle/input/stock-dataset/stock_dataset',
    metadata_csv=str(csv_path),
    model_family='deit',
    vit_checkpoint='facebook/deit-base-patch16-224',
    class_names=('down', 'up'),
    image_size=224,
    batch_size=16,
    epochs=12,
    patience=4,
    lr_backbone=3e-5,
    lr_head=2e-4,
    purge_gap=19,
)
vit_report = train_model(vit_cfg)
plot_history(vit_report)
plot_confusion(vit_report)
pd.DataFrame(vit_report['test']['classification_report']).T

## Alternative Recipe: Quantile Labels

If `binary_fixed` still underperforms, try balanced top/bottom quantile labels.


In [ ]:
quant_cfg = SignalConfig(
    data_root='/kaggle/input/stock-dataset/stock_dataset',
    output_csv='/kaggle/working/metadata_binary_quantile.csv',
    label_mode='binary_quantile',
    quantile=0.35,
    lookahead_days=1,
    purge_gap=19,
)
csv_path_q, summary_path_q = save_signal_metadata(quant_cfg)
print(csv_path_q)
pd.read_csv(csv_path_q)['label'].value_counts()